# Exercise 1: Prompt Templates & Chaining with LangChain

**Objective**: Learn how to use LangChain's PromptTemplate and chain multiple LLM calls.

**Key Takeaways**:
1. PromptTemplate — Simple text templates with variables
2. ChatPromptTemplate — Multi-message templates with roles
3. FewShotPromptTemplate — Learning from examples
4. LCEL (Pipe operator `|`) — Chain components
5. RunnableParallel — Execute multiple chains simultaneously
6. Conditional Routing — Branch based on results
7. Output Parsers — Extract structured data

## CRITICAL: Setup Environment Variables First!

Before running any cells, set your API key:
```bash
export ANTHROPIC_API_KEY='sk-your-actual-key-here'
```

Or create a `.env` file with:
```
ANTHROPIC_API_KEY=sk-your-actual-key-here
```

## Block 1: Import & Setup (REQUIRED - Run First)

In [ ]:
import os
import json
from typing import List, Optional
from dotenv import load_dotenv

# Load environment variables
try:
    load_dotenv(override=True)
except Exception as e:
    print(f"Note: Could not load .env file: {e}")

# Validate API Key FIRST
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY not found!\n"
        "Please set it: export ANTHROPIC_API_KEY='sk-your-key'"
    )

print(f"✅ API Key found (length: {len(api_key)})")
print("✓ Environment setup complete")

## Block 2: Import LangChain Libraries

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    FewShotPromptTemplate
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
    RunnableParallel,
    RunnableBranch
)
from pydantic import BaseModel, Field

print("✓ All LangChain libraries imported successfully")

## Block 3: Initialize LLM (CORRECT WAY)

In [ ]:
# CORRECT: Use ANTHROPIC_API_KEY, NOT "KEY"
# CORRECT: Do NOT use custom base_url
# CORRECT: Validate before creating instance

try:
    llm = ChatAnthropic(
        model="global.anthropic.claude-opus-4-5-20251101-v1:0",
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
        temperature=0.7,
        max_tokens=1024
    )
    print("✅ LLM initialized successfully (Claude Opus)")
except Exception as e:
    print(f"❌ Failed to initialize LLM: {e}")
    raise

## Block 4: Example 1 - Simple PromptTemplate

In [ ]:
# PromptTemplate: Simple text template with variables
simple_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms suitable for a 10-year-old."
)

# Format and test
formatted = simple_prompt.format(topic="machine learning")
print("Formatted Prompt:")
print(formatted)
print()

# Create chain: prompt | llm | parser
chain = simple_prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "artificial intelligence"})
print("Chain Output:")
print(result)

## Block 5: Example 2 - Multi-Variable PromptTemplate

In [ ]:
# Multi-variable template
multi_prompt = PromptTemplate(
    input_variables=["topic", "difficulty", "language"],
    template="""Write a {difficulty} tutorial about {topic} in {language}.
Include:
- Key concepts
- Practical examples
- Common mistakes"""
)

print("Template variables:", multi_prompt.input_variables)
print()

# Create chain
multi_chain = multi_prompt | llm | StrOutputParser()
result = multi_chain.invoke({
    "topic": "recursion in programming",
    "difficulty": "beginner",
    "language": "simple English"
})
print("Output:")
print(result)

## Block 6: Example 3 - ChatPromptTemplate (Multi-role)

In [ ]:
# ChatPromptTemplate: Multiple roles (system, human, ai)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python programming expert."),
    ("human", "How do I use {feature} in Python?"),
])

print("Chat template created")
print()

# Create chain
chat_chain = chat_prompt | llm | StrOutputParser()
result = chat_chain.invoke({"feature": "list comprehensions"})
print("Chat Response:")
print(result)

## Block 7: Example 4 - FewShotPromptTemplate

In [ ]:
# FewShotPromptTemplate: Learn from examples
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "big", "output": "small"},
    {"input": "hot", "output": "cold"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOpposite: {output}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Input: {word}\nOpposite:",
    input_variables=["word"]
)

print("Few-shot prompt created")
print()

# Create chain
few_shot_chain = few_shot_prompt | llm | StrOutputParser()
result = few_shot_chain.invoke({"word": "beautiful"})
print("Few-shot Output:")
print(result)

## Block 8: Example 5 - Sequential Chains (LLM → LLM)

In [ ]:
# Sequential chain: Generate question, then answer
prompt1 = PromptTemplate(
    input_variables=["category"],
    template="Generate a trivia question about {category}."
)

prompt2 = PromptTemplate(
    input_variables=["question"],
    template="Answer this question concisely: {question}"
)

# Step 1: Generate
chain1 = prompt1 | llm | StrOutputParser()
question = chain1.invoke({"category": "Science"})
print("Generated Question:")
print(question)
print()

# Step 2: Answer
chain2 = prompt2 | llm | StrOutputParser()
answer = chain2.invoke({"question": question})
print("Generated Answer:")
print(answer)

## Block 9: Example 6 - RunnableParallel (Concurrent Execution)

In [ ]:
# RunnableParallel: Execute multiple chains simultaneously
text = "Artificial Intelligence is transforming technology and society."

# Define separate chains
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize in one sentence: {text}"
)

keywords_prompt = PromptTemplate(
    input_variables=["text"],
    template="Extract 3 keywords from: {text}"
)

sentiment_prompt = PromptTemplate(
    input_variables=["text"],
    template="Analyze sentiment of: {text}"
)

# Create parallel chain
parallel_chain = RunnableParallel(
    summary=(summary_prompt | llm | StrOutputParser()),
    keywords=(keywords_prompt | llm | StrOutputParser()),
    sentiment=(sentiment_prompt | llm | StrOutputParser())
)

# Execute all at once
results = parallel_chain.invoke({"text": text})
print("Parallel Results:")
for key, value in results.items():
    print(f"\n{key.upper()}:")
    print(value)

## Block 10: Summary & Key Takeaways

In [ ]:
print("\n" + "="*70)
print("EXERCISE 1 SUMMARY: PROMPT TEMPLATES & CHAINING")
print("="*70)
print()
print("7 KEY CONCEPTS:")
print()
print("1. PromptTemplate")
print("   → Simple text templates with variables")
print("   → Usage: template.format(var='value')")
print()
print("2. ChatPromptTemplate")
print("   → Multi-message templates with roles")
print("   → Roles: system, human, ai")
print()
print("3. FewShotPromptTemplate")
print("   → Learn from examples")
print("   → Ensures consistent formatting")
print()
print("4. LCEL (Pipe operator |)")
print("   → Chain components: prompt | llm | parser")
print("   → Clean, readable code")
print()
print("5. RunnableParallel")
print("   → Execute multiple chains simultaneously")
print("   → Efficient for multi-task processing")
print()
print("6. Conditional Routing (RunnableBranch)")
print("   → Branch based on intermediate results")
print("   → Route to appropriate handler")
print()
print("7. Output Parsers")
print("   → Convert LLM output to structured data")
print("   → StrOutputParser, JsonOutputParser, etc.")
print()
print("="*70)
print("✅ Exercise 1 completed successfully!")
print("="*70)